# 08. Web Search Integration

An LLM only knows what it learned during training, so it can be **out of date**.
To answer about **today's** news or facts, the agent needs to **search the web**.

We give it a **web search tool** (a custom Python tool, just like notebook 07).
The agent's brain stays **OpenAI gpt-4o-mini**; the tool just fetches fresh web results.

## Real-life analogy

Even a smart person does not know everything by memory.
When they need fresh facts, they **Google it**.

A web search tool is the agent's **Google** — it looks things up, then the agent answers using what it found.

## What we use

| Piece | Job |
|-------|-----|
| `ddgs` (DuckDuckGo search) | Free web search, **no API key needed** |
| our `web_search` function | The tool the agent calls |
| gpt-4o-mini | Reads the results and writes the answer |

Install the free search library once:
```
pip install ddgs
```

In [2]:
from dotenv import load_dotenv
load_dotenv()

from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient

# A web search tool using DuckDuckGo (free, no API key).
def web_search(query: str) -> str:
    """Search the web for the query and return the top results as text."""
    from ddgs import DDGS
    results = DDGS().text(query, max_results=3)
    if not results:
        return "No results found."
    # Turn the results into simple text the agent can read
    return "\n\n".join(f"{r['title']}\n{r['body']}" for r in results)

model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")

searcher = AssistantAgent(
    name="searcher",
    model_client=model_client,
    tools=[web_search],                 # the web search tool
    reflect_on_tool_use=True,
    system_message="You answer questions using the web_search tool. Keep answers short.",
)

result = await searcher.run(task="Who won the latest FIFA World Cup? Search the web.")
print(result.messages[-1].content)

The latest FIFA World Cup was held in 2022, and Argentina won the tournament.


## Key points to remember

- LLMs can be **out of date**; web search gives them **fresh facts**.
- A **web search tool** is just a custom Python tool (notebook 07 pattern).
- We use **`ddgs`** (DuckDuckGo) because it is **free and needs no API key**.
- The **brain stays OpenAI gpt-4o-mini** — the tool only fetches results.
- Give it to the agent with `tools=[web_search]` and `reflect_on_tool_use=True`.